# 1.8 合成数据验证 (Synthetic Data Verification)

> 🕐 预估学习时间：35分钟

合成数据便宜但易塌缩、泄漏与风格单一。产业流程强调：生成 → 自动验证 → 多样性过滤 → 人工抽检。

本节涵盖：
- 常见失效模式
- 执行/模式/裁判验证器
- 去重与多样性（embedding n-gram）
- 泄漏检测（相对训练语料）
- 质量门禁流水线


## 1. 失效模式

| 模式 | 现象 | 后果 |
|------|------|------|
| 模式塌缩 | 很多样本几乎同构 | 微调后不会泛化 |
| 伪正确 | 看起来对，逻辑错 | 奖励黑客 |
| 泄漏 | 复述评测集 | 虚高榜 |
| 有害漂移 | 安全边界被冲淡 | 上线风险 |


In [ ]:
import re
import torch
import torch.nn.functional as F
from collections import Counter

torch.manual_seed(0)


def is_schema_ok(sample: dict) -> bool:
    return isinstance(sample.get('instruction'), str) and isinstance(sample.get('output'), str) and len(sample['output']) > 0


def is_code_executable(output: str) -> bool:
    # only allow simple pure functions for the toy verifier
    if 'import os' in output or 'open(' in output:
        return False
    try:
        ns = {}
        exec(output, ns, ns)
        if 'add' in ns:
            return ns['add'](1, 2) == 3
        return True
    except Exception:
        return False


samples = [
    {'instruction': 'write add', 'output': 'def add(a,b):\n    return a+b\n'},
    {'instruction': 'write add', 'output': 'def add(a,b):\n    return a-b\n'},
    {'instruction': 'x', 'output': ''},
]
print('=== Verifiers ===')
for s in samples:
    print(s['output'][:30].replace('\n',' '), 'schema=', is_schema_ok(s), 'exec=', is_code_executable(s['output']))
print(f'Key: Task-specific verifiers catch silent errors that LLM-as-judge may miss.')


## 2. 多样性与近重复过滤

用 n-gram Jaccard 或嵌入相似度剔除近重复，保持指令覆盖面。


In [ ]:
def trigrams(text: str) -> set[str]:
    toks = text.lower().split()
    return set(' '.join(toks[i:i+3]) for i in range(max(len(toks)-2, 1)))


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / max(len(a | b), 1)


def dedup(texts, thr=0.6):
    kept = []
    grams = []
    for t in texts:
        g = trigrams(t)
        if any(jaccard(g, kg) >= thr for kg in grams):
            continue
        kept.append(t)
        grams.append(g)
    return kept


texts = [
    'explain gradient descent step by step',
    'explain gradient descent steps carefully',
    'how does attention work in transformers',
    'explain gradient descent step by step please',
]
kept = dedup(texts, thr=0.4)
print('=== Diversity Filter ===')
print('kept:', kept)
print(f'removed={len(texts)-len(kept)}')
print(f'\nKey: Near-duplicate synthetic instructions waste epochs and amplify mode collapse.')


## 3. 评测集泄漏检查

对合成输出与基准题干做 n-gram / 嵌入重叠检测，超过阈值则丢弃或隔离。


In [ ]:
BENCHMARK = [
    'what is the capital of france',
    'write a function to reverse a linked list',
]


def leak_score(text: str, bench=BENCHMARK) -> float:
    g = trigrams(text)
    return max(jaccard(g, trigrams(b)) for b in bench)


cands = [
    'Describe PagedAttention briefly',
    'write a function to reverse a linked list in python',
]
print('=== Leak Check ===')
for c in cands:
    print(c, 'leak=', round(leak_score(c), 3))
print(f'\nKey: Filter synthetic data against eval n-grams before mixing into SFT.')


## 4. 质量门禁流水线

`generate → schema → verifier → dedup → leak → safety → accept`

统计每关通过率，定位生成提示或教师模型问题。


In [ ]:
def pipeline(raw_samples):
    stats = Counter()
    accepted = []
    for s in raw_samples:
        stats['total'] += 1
        if not is_schema_ok(s):
            stats['fail_schema'] += 1
            continue
        if s['instruction'].startswith('write') and not is_code_executable(s['output']):
            stats['fail_exec'] += 1
            continue
        if leak_score(s['instruction'] + ' ' + s['output']) > 0.5:
            stats['fail_leak'] += 1
            continue
        accepted.append(s)
        stats['accepted'] += 1
    # dedup on instructions
    instr = [s['instruction'] for s in accepted]
    kept_instr = set(dedup(instr, thr=0.5))
    accepted = [s for s in accepted if s['instruction'] in kept_instr]
    stats['after_dedup'] = len(accepted)
    return accepted, stats


raw = samples + [
    {'instruction': 'write a function to reverse a linked list', 'output': 'def add(a,b):\n    return a+b\n'},
    {'instruction': 'explain attention', 'output': 'Attention mixes tokens by similarity.'},
]
accepted, stats = pipeline(raw)
print('=== Gate Pipeline ===')
print(dict(stats))
print('accepted=', accepted)
print(f'\nKey: Measurable gates turn synthetic data from lottery tickets into an engineered supply chain.')


## 课后思考题

1. 什么时候该用执行验证，什么时候只能用 LLM-as-judge？
2. 多样性过滤过严会伤害哪些长尾能力？
3. 如何防止教师模型把自己的偏见放大到学生模型？
4. 合成数据占比升高时，如何监控真实用户分布偏移？

---
> 本节涵盖了1.8 合成数据验证的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
